In [5]:
# Installing Unsloth: optimized fine-tuning library for LLMs
# Installing:
# TRL (trainer)
# Transformers (model loading)
# Datasets (data handling)
# Accelerate (GPU support)
# BitsAndBytes (4-bit quantization)
!pip install unsloth
!pip install trl transformers datasets accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224

In [6]:
import torch                                             # Import PyTorch (deep learning framework where everything runs on)

device = "cuda" if torch.cuda.is_available() else "cpu"  # Check if a GPU (CUDA) is available, if not, fall back to CPU
print(f"Verwendetes Gerät: {device}")                    # Print which device is being used

if device == "cuda":                                     # Only run the next lines if a GPU was found
    print(f"GPU: {torch.cuda.get_device_name(0)}")       # Print the GPU model name (e.g., "Tesla T4")
    print(f"VRAM verfügbar: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")  # Print total GPU memory in GB

Verwendetes Gerät: cuda
GPU: Tesla T4
VRAM verfügbar: 15.6 GB


In [7]:
from unsloth import FastLanguageModel        # Import Unsloth's optimized model loader

max_seq_length = 2048                                            # Maximum number of tokens the model can process at once (2048 is enough for tax law)

model, tokenizer = FastLanguageModel.from_pretrained(            # Load the pre-trained model and its tokenizer together
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",  # Use Meta's Llama 3.1 8B instruct model, pre-quantized to 4-bit by Unsloth
    max_seq_length = max_seq_length,                             # Pass the max token length defined above
    dtype = None,                                                # Let Unsloth automatically pick the best data type (e.g., float16 or bfloat16)
    load_in_4bit = True,                                         # Load model weights in 4-bit precision (QLoRA) — drastically reduces VRAM usage
)

print("Modell erfolgreich geladen!")         # Confirm the model loaded successfully

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


Modell erfolgreich geladen!


In [8]:
model = FastLanguageModel.get_peft_model(             # Wrap the model with LoRA adapters (PEFT = Parameter-Efficient Fine-Tuning)
    model,                                            # base model (loaded in the previous cell)
    r = 16,                                           # LoRA rank: controls how many new parameters are added, 16 is balanced default
    target_modules = [                                # List of transformer sub-layers to attach LoRA adapters to
        "q_proj",                                     # Query projection matrix in the attention mechanism
        "k_proj",                                     # Key projection matrix in the attention mechanism
        "v_proj",                                     # Value projection matrix in the attention mechanism
        "o_proj",                                     # Output projection after the attention heads
        "gate_proj",                                  # Gate layer inside the feed-forward network (FFN)
        "up_proj",                                    # Up-projection inside the FFN
        "down_proj"                                   # Down-projection inside the FFN
    ],
    lora_alpha = 16,                                  # Scaling factor for the LoRA updates; usually set equal to r
    lora_dropout = 0,                                 # Dropout rate for LoRA layers; 0 = disabled (recommended by Unsloth for speed)
    bias = "none",                                    # Do not train any bias parameters — keeps training minimal
    use_gradient_checkpointing = "unsloth",           # Use Unsloth's gradient checkpointing to save VRAM at a small speed cost
    random_state = 42,                                # Fix the random seed so results are reproducible
)

# Shows how many parameters are actually being trained
model.print_trainable_parameters()                    # Print how many parameters will actually be updated during training (should be ~0.5% of total)

Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [11]:
import json                                            # Standard library for reading JSON files
from datasets import Dataset                           # Hugging Face Dataset class for structured data handling

# JSON laden
with open("drive/MyDrive/Dp4_Modelle/austrian_tax_law_training_dataset.json", "r", encoding="utf-8") as f:  # Open the training JSON file from Google Drive with UTF-8 encoding
    raw = json.load(f)                                # Parse the entire JSON file into a Python dictionary

data = raw["training_data"]                           # Extract only the "training_data" list from the JSON (ignoring metadata etc.)

print(f"Anzahl Trainingsbeispiele: {len(data)}")      # Print how many training examples are in the dataset
print(f"Erster Eintrag zur Kontrolle:")               # Label for the inspection output below
print(f"  ID: {data[0]['id']}")                       # Print the ID of the first training example
print(f"  Prompt: {data[0]['prompt'][:80]}...")       # Print the first 80 characters of the first question

# Alpaca Prompt Template
alpaca_prompt = """### Instruction:
{}

### Input:
{}

### Response:
{}"""                                                  # Define the Alpaca prompt format: Instruction + Input question + Expected response (three {} placeholders)

EOS_TOKEN = tokenizer.eos_token                        # Get the end-of-sequence token string from the tokenizer (e.g., "<|eot_id|>")

INSTRUCTION = "Du bist ein Experte für österreichisches Steuerrecht. Beantworte die folgende Frage präzise mit Angabe der relevanten Gesetzesparagraphen."  # Fixed system instruction that gets placed in the "Instruction" slot of every training example

def formatting_prompts_func(examples):                # Define a function that formats a batch of examples into the Alpaca prompt structure
    texts = []                                        # Initialize an empty list to collect formatted strings
    for prompt, answer in zip(examples["prompt"], examples["correct_answer"]):  # Iterate over question-answer pairs in the batch
        text = alpaca_prompt.format(
            INSTRUCTION,                              # Fill in the fixed instruction text
            prompt,                                   # Fill in the question from the dataset's "prompt" field
            answer                                    # Fill in the correct answer from the dataset's "correct_answer" field
        ) + EOS_TOKEN                                 # Append the end-of-sequence token so the model knows where to stop
        texts.append(text)                            # Add the formatted text to the list
    return {"text": texts}                            # Return as a dictionary with key "text" (required by the Dataset API)

# Creating Dataset
dataset = Dataset.from_list(data)                     # Convert the raw list of dicts into a Hugging Face Dataset object
dataset = dataset.map(formatting_prompts_func, batched=True)  # Apply the formatting function to all examples in batches

print(f"\nDataset erfolgreich erstellt mit {len(dataset)} Einträgen")  # Confirm the dataset was created successfully
print("\nVorschau erster Trainingstext:")             # Label for the preview below
print(dataset[0]["text"][:500])                       # Print the first 500 characters of the first formatted training text

Anzahl Trainingsbeispiele: 253
Erster Eintrag zur Kontrolle:
  ID: ESTG-1-001
  Prompt: Wer unterliegt in Österreich der unbeschränkten Einkommensteuerpflicht?...


Map:   0%|          | 0/253 [00:00<?, ? examples/s]


Dataset erfolgreich erstellt mit 253 Einträgen

Vorschau erster Trainingstext:
### Instruction:
Du bist ein Experte für österreichisches Steuerrecht. Beantworte die folgende Frage präzise mit Angabe der relevanten Gesetzesparagraphen.

### Input:
Wer unterliegt in Österreich der unbeschränkten Einkommensteuerpflicht?

### Response:
Der unbeschränkten Einkommensteuerpflicht unterliegen jene natürlichen Personen, die im Inland einen Wohnsitz oder ihren gewöhnlichen Aufenthalt haben. Die unbeschränkte Steuerpflicht erstreckt sich auf alle in- und ausländischen Einkünfte (Welt


In [12]:
from trl import SFTTrainer                            # Import the Supervised Fine-Tuning Trainer from TRL
from transformers import TrainingArguments            # Import the training configuration class

trainer = SFTTrainer(                                 # Create a fine-tuning trainer object
    model = model,                                    # The LoRA-wrapped model to train
    tokenizer = tokenizer,                            # The tokenizer to use for encoding the text
    train_dataset = dataset,                          # The formatted dataset to train on
    dataset_text_field = "text",                      # Name of the field in the dataset that contains the text to train on
    max_seq_length = max_seq_length,                  # Maximum sequence length (2048 tokens, set earlier)
    dataset_num_proc = 2,                             # Number of parallel processes for dataset preprocessing
    args = TrainingArguments(                         # Define all training hyperparameters
        per_device_train_batch_size = 2,              # Process 2 examples per GPU per step
        gradient_accumulation_steps = 4,              # Accumulate gradients over 4 steps before updating weights; effective batch size = 2×4 = 8
        warmup_steps = 5,                             # Gradually increase the learning rate for the first 5 steps to stabilize early training
        num_train_epochs = 3,                         # Train for 3 full passes through the dataset
        learning_rate = 2e-4,                         # Step size for weight updates (0.0002)
        fp16 = not torch.cuda.is_bf16_supported(),    # Use float16 precision if bfloat16 is NOT supported by the GPU
        bf16 = torch.cuda.is_bf16_supported(),        # Use bfloat16 precision if the GPU supports it (T4 does not, so fp16 is used)
        logging_steps = 1,                            # Log the training loss after every single step
        optim = "adamw_8bit",                         # Use 8-bit AdamW optimizer — memory-efficient version of the standard optimizer
        weight_decay = 0.01,                          # L2 regularization to reduce overfitting
        lr_scheduler_type = "linear",                 # Linearly decrease the learning rate from its peak to zero over training
        output_dir = "outputs",                       # Directory where model checkpoints are saved
        report_to = "none",                           # Disable reporting to external loggers like Weights & Biases or TensorBoard
    ),
)

print("Training startet...")                          # Announce that training is about to begin
trainer_stats = trainer.train()                       # Start the training loop and store the resulting metrics
print(f"\nTraining abgeschlossen!")                   # Announce training finished
print(f"Trainingszeit: {trainer_stats.metrics['train_runtime']:.0f} Sekunden")  # Print total training duration in seconds
print(f"Finaler Loss: {trainer_stats.metrics['train_loss']:.4f}")  # Print the final average training loss (lower = better fit)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/253 [00:00<?, ? examples/s]

Training startet...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 253 | Num Epochs = 3 | Total steps = 96
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.880406
2,1.824155
3,1.777035
4,1.683219
5,1.637912
6,1.452764
7,1.476132
8,1.261120
9,1.253223
10,1.222790



Training abgeschlossen!
Trainingszeit: 615 Sekunden
Finaler Loss: 0.8595


In [13]:
FastLanguageModel.for_inference(model)                # Switch the model to inference mode (disables training-specific features, speeds up generation)

# Testfrage
test_prompt = alpaca_prompt.format(                       # Build a test prompt using the same Alpaca format used during training
    "Du bist ein Experte für österreichisches Steuerrecht. Beantworte die folgende Frage präzise mit Angabe der relevanten Gesetzesparagraphen.",  # Same fixed instruction
    "Was sind Einkünfte aus Gewerbebetrieb gemäß EStG?",  # The test question about business income under Austrian tax law
    ""                                                    # Leave the response slot empty — the model will fill this in
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")  # Tokenize the prompt into integer IDs and move the tensor to the GPU

outputs = model.generate(                             # Run the model to generate a response
    **inputs,                                         # Unpack the tokenized input (input_ids and attention_mask)
    max_new_tokens = 300,                             # Generate at most 300 new tokens
    use_cache = True,                                 # Use the key-value cache to speed up generation
    pad_token_id = tokenizer.eos_token_id,            # Use the EOS token as the padding token to avoid warnings
)

# Only extracting the generated text (not repeating the promt)
generated_text = tokenizer.decode(                    # Decode the generated token IDs back into human-readable text
    outputs[0][inputs["input_ids"].shape[1]:],        # Slice off the prompt tokens — only decode the newly generated part
    skip_special_tokens=True                          # Remove special tokens (like EOS) from the decoded output
)

print("Modellantwort:")                               # Label for the output
print(generated_text)                                 # Print the model's generated answer

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Modellantwort:
Einkünfte aus Gewerbebetrieb gemäß § 23 EStG sind Einkünfte, die einem Gewerbebetrieb (Betrieb einer Gewerbebetriebes im Sinne des Gewerbeordnungstextes) entstammen. Dazu zählen Einkünfte aus selbständiger Arbeit im Gewerbebetrieb (§ 22 EStG), Einkünfte aus Gewerbebetrieb einer Personengesellschaft (GmbH, OG, KG) und Einkünfte aus Kapitalgesellschaften (GmbH, AG). Gewinne aus der Veräußerung von Betriebsvorrichtungen und Betriebsaufzügen können ebenfalls als Gewerbebetriebseinkünfte qualifiziert werden.


In [14]:
import pandas as pd                                   # Import pandas for working with tabular data (CSV, DataFrames)

# Loading dataset from Github
df = pd.read_csv(                                     # Load a CSV file directly into a DataFrame
    "https://raw.githubusercontent.com/svakulenk0/wu-llms-ss26/main/dataset_clean.csv"  # URL of the test dataset on GitHub
)

print(f"Test-Dataset geladen: {len(df)} Fragen")      # Print how many questions are in the test dataset
print(df.head(2))                                     # Display the first 2 rows as a sanity check

results = []                                          # Initialize an empty list to collect generated answers

for idx, row in df.iterrows():                        # Iterate over every row (question) in the test dataset

    # Formating promt for the question
    prompt = alpaca_prompt.format(                    # Build the Alpaca-formatted prompt for this question
        "Du bist ein Experte für österreichisches Steuerrecht. Beantworte die folgende Frage präzise mit Angabe der relevanten Gesetzesparagraphen.",  # Fixed instruction
        row["prompt"],                                # The question from this row of the CSV
        ""                                            # Empty response slot — to be generated by the model
    )

    # Tokenizing and onto GPU
    inputs = tokenizer(                               # Tokenize the formatted prompt
        prompt,
        return_tensors="pt",                          # Return PyTorch tensors
        truncation=True,                              # Truncate if the prompt is too long
        max_length=1800                               # Limit prompt to 1800 tokens to leave room for the 300-token response
    ).to("cuda")                                      # Move the tokenized input to the GPU

    # Generating answers
    with torch.no_grad():                             # Disable gradient computation — not needed during inference, saves memory
        output = model.generate(                      # Generate the model's response
            **inputs,                                 # Pass tokenized input
            max_new_tokens = 300,                     # Generate up to 300 new tokens
            use_cache = True,                         # Enable KV-cache for faster decoding
            pad_token_id = tokenizer.eos_token_id,    # Use EOS token as padding to suppress warnings
            temperature = 0.1,                        # Low temperature = more deterministic, less random output
            do_sample = True,                         # Enable sampling (required when temperature is set)
        )

    # Only decoding the generated text (without the promt)
    answer = tokenizer.decode(                        # Decode generated tokens back to text
        output[0][inputs["input_ids"].shape[1]:],     # Skip the prompt tokens, only decode the new ones
        skip_special_tokens=True                      # Remove special tokens from the output
    ).strip()                                         # Remove leading/trailing whitespace

    results.append({                                  # Add this result to the list
        "id": row["id"],                              # Include the question's ID for identification
        "answer": answer                              # Include the model's generated answer
    })

    # Showing progress
    if idx % 50 == 0:                                 # Every 50 questions...
        print(f"Fortschritt: {idx}/{len(df)} Fragen bearbeitet")  # Print a progress update

print(f"\nFertig! {len(results)} Antworten generiert.")  # Announce that all questions have been answered

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test-Dataset geladen: 643 Fragen
             id                                             prompt
0  CORP-TAX-001  Was ist die steuerliche Bemessungsgrundlage fü...
1  CORP-TAX-002  Welche steuerlichen Konsequenzen hat es, wenn ...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fortschritt: 0/643 Fragen bearbeitet


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

Fortschritt: 550/643 Fragen bearbeitet


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Fortschritt: 600/643 Fragen bearbeitet


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


Fertig! 643 Antworten generiert.


In [15]:
# Creating a DataFrame
results_df = pd.DataFrame(results)                    # Convert the list of result dicts into a pandas DataFrame

# Checking how it looks
print("Erste 3 Einträge:")                            # Label for the preview
print(results_df.head(3))                             # Display the first 3 rows (id + answer)
print(f"\nGesamtanzahl Zeilen: {len(results_df)}")    # Print total number of rows

# Checking if all questions have been answered
assert len(results_df) == len(df), "Achtung: Nicht alle Fragen beantwortet!"  # Crash with an error message if the count doesn't match the test dataset

# Saving the CSV
results_df.to_csv(                                    # Save the DataFrame as a CSV file
    "ft_model_output.csv",                            # File name for the output
    index=False,                                      # Do not write row numbers as a column
    sep=","                                           # Use comma as the column separator
)

print("\nCSV gespeichert: ft_model_output.csv")       # Confirm the file was saved

# Downloading
from google.colab import files                        # Import Google Colab's file download utility
files.download("ft_model_output.csv")                 # Trigger a browser download of the CSV file to the local machine

Erste 3 Einträge:
             id                                             answer
0  CORP-TAX-001  Die Bemessungsgrundlage für die Körperschaftst...
1  CORP-TAX-002  Zinslose Darlehen von einer Kapitalgesellschaf...
2  CORP-TAX-003  Gemäß § 7 Abs. 3 Z 2 EStG sind folgende Körper...

Gesamtanzahl Zeilen: 643

CSV gespeichert: ft_model_output.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>